# Pipeline : Mapping, SNP Calling and SDPOP for *Laurus azorica* on the 12 CHR

# 1. Mapping and SNP calling

## 1.0 Before running the WORKFLOW

In [ ]:
# Create the directories
mkdir /home/tbertrand/work/L_nobilis_all_12_CHR
mkdir /home/tbertrand/work/L_nobilis_all_12_CHR/GENOME
mkdir /home/tbertrand/work/L_nobilis_all_12_CHR/INPUT
mkdir /home/tbertrand/work/L_nobilis_all_12_CHR/MAPPING
mkdir /home/tbertrand/work/L_nobilis_all_12_CHR/GENOTYPING
mkdir /home/tbertrand/work/L_nobilis_all_12_CHR/WORKFLOW_MAPPING_FREEBAYES

# Download the fasta_generate_regions.py script 
cd /home/tbertrand/work/L_nobilis_all_12_CHR/GENOTYPING #ATTENTION A BIEN PLACER LE SCRIPT DANS CE DIRECTORY
wget https://raw.githubusercontent.com/freebayes/freebayes/master/scripts/fasta_generate_regions.py

# Create the input list in the following format (Tab separated)  
R1.fasta    R2.fasta    Indiv_name

# Exemple: 
/home/tbertrand/work/RNA_data/trimming/LN-T1_Mb_R1_001_val_1.fq.gz  /home/tbertrand/work/RNA_data/trimming/LN-T1_Mb_R2_001_val_2.fq.gz	LN-T1_M
/home/tbertrand/work/RNA_data/trimming/LN-T2_F_R1_001_val_1.fq.gz	/home/tbertrand/work/RNA_data/trimming/LN-T2_F_R2_001_val_2.fq.gz	LN-T2_F
/home/tbertrand/work/RNA_data/trimming/LN-T3_F_R1_001_val_1.fq.gz	/home/tbertrand/work/RNA_data/trimming/LN-T3_F_R2_001_val_2.fq.gz	LN-T3_F
/home/tbertrand/work/RNA_data/trimming/LN-T3_Female_R1_001_val_1.fq.gz	/home/tbertrand/work/RNA_data/trimming/LN-T3_Female_R2_001_val_2.fq.gz	LN-T3_F
/home/tbertrand/work/RNA_data/trimming/LN-T4_M_R1_001_val_1.fq.gz	/home/tbertrand/work/RNA_data/trimming/LN-T4_M_R2_001_val_2.fq.gz	LN-T4_M

# Save it in the INPUT directory
"/home/tbertrand/work/L_novocanariensis/INPUT/reads_list_novocanariensis.txt"

## 1.0.1 Remove the unplaced scaffolds on the L. azorica ref genome

In [ ]:
#On recupere les 12 CHR principaux
grep "^OZ" GCA_977007225.1_dmLauAzor1.pri_genomic.fna.fai | cut -f1 > chr12.list

#On les extraits
samtools faidx GCA_977007225.1_dmLauAzor1.pri_genomic.fna -r chr12.list > GCA_977007225.1_dmLauAzor1_12_CHR.pri_genomic.fna

#On indexe le genome restreind
samtools faidx GCA_977007225.1_dmLauAzor1_12_CHR.pri_genomic.fna

##  1.1 Indexing the reference genome with STAR

In [ ]:
#!/bin/bash
#SBATCH --job-name=index
#SBATCH -p unlimitq
#SBATCH --time=7-00:00:00
#SBATCH --mail-user=tristan.bertrand@umontpellier.fr
#SBATCH --mail-type=ALL
#SBATCH --mem-per-cpu=8G
#SBATCH --cpus-per-task=24

#############################
# 1. MODULES
#############################

module load bioinfo/STAR/2.7.11b

#############################
# 2. FILES 
#############################

WORKDIR=/home/tbertrand/work/L_nobilis_all_12_CHR/GENOME/Index
FASTA=/home/tbertrand/work/L_nobilis_all_12_CHR/GENOME/GCA_977007225.1_dmLauAzor1_12_CHR.pri_genomic.fna
ANNOTATION=/home/tbertrand/work/L_nobilis_all_12_CHR/GENOME/braker.fixed_ids.gff3

#############################
# 3. RUN STAR genomeGenerate
#############################

STAR --runMode genomeGenerate --genomeDir ${WORKDIR}  --genomeFastaFiles ${FASTA}  --runThreadN 24 --sjdbGTFfile ${ANNOTATION}  --sjdbGTFtagExonParentTranscript Parent

##  1.1 PIPELINE MAPPING & SNPCALLING

snakefile

In [ ]:
configfile: "config.yaml"

# A METTRE APRES AVOIR FAIT wc -l genotyping_dir/regions/all_regions.bed
NREGIONS = 13338
regions = list(range(1, NREGIONS + 1))

# Lire table samples
samples = {}
with open(config["sample_table"]) as f:
    for line in f:
        r1, r2, ind = line.strip().split()
        samples[ind] = {"r1": r1, "r2": r2}

INDIVIDUALS = list(samples.keys())


rule all:
    input:
        # tous les BAM traités pour freebayes
        expand(config["mapping_dir"] + "/{ind}_TranscriptomeGenomePos.sorted.rg.bam", ind=INDIVIDUALS),
        # le fichier BAM_LIST.txt pour freebayes 
        config["genotyping_dir"] + "/BAM_LIST.txt",
        # Tous les VCF chunks générés (pour que Snakemake relance les manquants)
        expand(config["genotyping_dir"] + "/vcf_chunks/variants.{i}.vcf", i=regions),
        # VCF final concaténé
        config["genotyping_dir"] + "/L_nobilis_all_Lazorica_VCFALLSITES_C2_REPMONO_NBESTALLELES2.vcf"
        

#################################
# STAR mapping
#################################

rule star_map:
    input:
        r1=lambda wc: samples[wc.ind]["r1"],
        r2=lambda wc: samples[wc.ind]["r2"]
    output:
        coord=config["mapping_dir"] + "/{ind}_Aligned.sortedByCoord.out.bam",
        transcriptome=config["mapping_dir"] + "/{ind}_Aligned.toTranscriptome.out.bam"
    threads: 8
    resources:
        mem_mb=96000,
        star_slots=1,
        runtime = 2000
    shell:
        """
        module load bioinfo/STAR/2.7.11b

        STAR \
          --genomeDir {config[genome_dir]} \
          --readFilesIn {input.r1} {input.r2} \
          --readFilesCommand gunzip -c \
          --runThreadN {threads} \
          --quantMode TranscriptomeSAM \
          --outFileNamePrefix {config[mapping_dir]}/{wildcards.ind}_ \
          --outSAMtype BAM SortedByCoordinate
        """


#################################
# BAM processing
#################################

rule process_bam:
    input:
        coord=config["mapping_dir"] + "/{ind}_Aligned.sortedByCoord.out.bam",
        transcriptome=config["mapping_dir"] + "/{ind}_Aligned.toTranscriptome.out.bam"
    output:
        final=config["mapping_dir"] + "/{ind}_TranscriptomeGenomePos.sorted.rg.bam"
    threads: 4
    resources:
        mem_mb=32000,
        process_bam_slots=1,
        runtime = 1000
    shell:
        """
        module load bioinfo/samtools/1.23

        samtools sort -@ {threads} {input.transcriptome} \
            -o {config[mapping_dir]}/{wildcards.ind}_Aligned.toTranscriptome.sorted.out.bam

        samtools flagstat {config[mapping_dir]}/{wildcards.ind}_Aligned.toTranscriptome.sorted.out.bam \
            > {config[mapping_dir]}/{wildcards.ind}_Aligned.toTranscriptome.sorted.flagstat

        samtools view {config[mapping_dir]}/{wildcards.ind}_Aligned.toTranscriptome.sorted.out.bam | cut -f1 \
            > {config[mapping_dir]}/{wildcards.ind}_transcriptome_reads.txt

        samtools view -N {config[mapping_dir]}/{wildcards.ind}_transcriptome_reads.txt -bh {input.coord} \
            > {config[mapping_dir]}/{wildcards.ind}_TranscriptomeGenomePos.bam

        samtools addreplacerg \
            -r ID:{wildcards.ind} \
            -r SM:{wildcards.ind} \
            -r PL:ILLUMINA \
            -r LB:{wildcards.ind} \
            -r PU:{wildcards.ind} \
            -o {config[mapping_dir]}/{wildcards.ind}_TranscriptomeGenomePos.rg.bam \
            {config[mapping_dir]}/{wildcards.ind}_TranscriptomeGenomePos.bam

        samtools sort -@ {threads} \
            -o {output.final} \
            {config[mapping_dir]}/{wildcards.ind}_TranscriptomeGenomePos.rg.bam

        samtools index {output.final}

        rm -f {input.coord}
        rm -f {input.transcriptome}
        rm -f {config[mapping_dir]}/{wildcards.ind}_Aligned.toTranscriptome.sorted.out.bam
        rm -f {config[mapping_dir]}/{wildcards.ind}_TranscriptomeGenomePos.bam
        rm -f {config[mapping_dir]}/{wildcards.ind}_TranscriptomeGenomePos.rg.bam
        rm -f {config[mapping_dir]}/{wildcards.ind}_transcriptome_reads.txt
        """

##################################
# règle pour générer BAM_LIST.txt
##################################

rule make_bam_list:
    input:
        expand(config["mapping_dir"] + "/{ind}_TranscriptomeGenomePos.sorted.rg.bam", ind=INDIVIDUALS)
    output:
        config["genotyping_dir"] + "/BAM_LIST.txt"
    run:
        with open(output[0], "w") as out:
            for bam in input:
                out.write(bam + "\n")

###########################################
# RULE : Generate Freebayes regions
###########################################

rule GenerateFreebayesRegions:
    input:
        index=config["ref_genome"] + ".fai"
    output:
        expand(config["genotyping_dir"] + "/regions/region.{i}.bed", i=regions)
    shell:
        """
        module load bioinfo/freebayes/1.3.6
        module load devel/Miniforge/Miniforge3
        module load bioinfo/vcflib/1.0.12
        mkdir -p {config[genotyping_dir]}/regions

        python {config[genotyping_dir]}/fasta_generate_regions.py \
        {input.index} 100000 \
        > {config[genotyping_dir]}/regions/all_regions.bed

        awk '{{ 
            gsub(":", "\\t"); 
            gsub("-", "\\t"); 
            print > "{config[genotyping_dir]}/regions/region." NR ".bed" 
        }}' {config[genotyping_dir]}/regions/all_regions.bed
        """

###########################################
# RULE : Freebayes avec retries 
###########################################

def get_mem_mb(wildcards, attempt):
    """
    Retourne la mémoire en MB selon le numéro de tentative.
    Attempt=1 → 10 GB
    Attempt=2 → 50 GB
    Attempt=3 → 100 GB
    Attempt=4 → 200 GB

    """
    if attempt == 1:
        return 10000
    elif attempt == 2:
        return 50000
    elif attempt == 3:
        return 100000
    else:
        return 200000

rule VariantCallingFreebayes:
    input:
        ref     = config["ref_genome"],
        regions = config["genotyping_dir"] + "/regions/region.{i}.bed",
        bamlist = config["genotyping_dir"] + "/BAM_LIST.txt"
    output:
        temp(config["genotyping_dir"] + "/vcf_chunks/variants.{i}.vcf")
    log:
        config["genotyping_dir"] + "/logs/freebayes_region.{i}.log"
    threads: 1
    retries: 3
    resources:
        mem_mb = get_mem_mb,
        runtime = 2880
    shell:
        """
	echo "Memory: {resources.mem_mb} MB" >> {log}
        module load bioinfo/freebayes/1.3.6
        module load devel/Miniforge/Miniforge3
        module load bioinfo/vcflib/1.0.12

        mkdir -p {config[genotyping_dir]}/vcf_chunks

        freebayes \
          -f {input.ref} \
          -t {input.regions} \
          -L {input.bamlist} \
          -C 2 \
           -g 1500000 \ 
          --report-monomorphic \
          --use-best-n-alleles 2 \
          > {output} 2>> {log}
        """


###########################################
# RULE : Concatenate VCF
###########################################

rule ConcatVCFs:
    input:
        calls=expand(config["genotyping_dir"] + "/vcf_chunks/variants.{i}.vcf", i=regions)
    output:
        config["genotyping_dir"] + "/L_nobilis_all_Lazorica_VCFALLSITES_C2_REPMONO_NBESTALLELES2.vcf"
    log:
        "logs/ConcatVCFs.log"
    threads: 4
    shell:
        """
        module load bioinfo/Bcftools/1.9

        bcftools concat {input.calls} -Ov -o {output}
        """

config file

In [ ]:
genome_dir: "/home/tbertrand/work/L_nobilis_all_12_CHR/GENOME/Index"
ref_genome: "/home/tbertrand/work/L_nobilis_all_12_CHR/GENOME/Index/GCA_977007225.1_dmLauAzor1_12_CHR.pri_genomic.fna"
sample_table: "/home/tbertrand/work/L_nobilis_all_12_CHR/INPUT/reads_list_L_nobilis_all.txt"
mapping_dir: "/home/tbertrand/work/L_nobilis_all_12_CHR/MAPPING"
genotyping_dir: "/home/tbertrand/work/L_nobilis_all_12_CHR/GENOTYPING"

run_snakefile.sh

In [ ]:
#!/bin/bash
#SBATCH --job-name=Snakemake_L_nobilis_all_12_CHR
#SBATCH -p unlimitq
#SBATCH --time=20-00:00:00
#SBATCH --mail-user=tristan.bertrand@umontpellier.fr
#SBATCH --mail-type=ALL
#SBATCH --cpus-per-task=1
#SBATCH --mem=8G
#SBATCH --output=snakemake_%j.log

module load bioinfo/Snakemake/8.20.3

snakemake \
  --snakefile snakefile_freebayes \
  --jobs 64 \
  --rerun-incomplete \
  --keep-going \
  --latency-wait 60 \
  --executor slurm  \
  --default-resources mem_mb=8000 -n -p

##  2 SDPOP

## 2.0 Before running SDpop

0. Create directory

In [ ]:
mkdir SD_POP
mkdir WORKFLOW_SDPOP

1. create file BED_EXON

In [ ]:
# creation BED_EXON
cd /home/tbertrand/work/L_nobilis_all_12_CHR/GENOME

grep "exon" braker.fixed_ids.gff3 \
| awk 'BEGIN{OFS="\t"}{
    match($9,/gene_id=([^;]+)/,a);
    print $1,$4-1,$5,a[1]
}' \
| sort -k1,1 -k2,2n \
> braker.fixed_ids_exons_sorted.bed

2. Select the individulals to filter from the vcf

In [ ]:
# Mauvais mapping
Lm-M02Cb
Lm-M03Cb
Lm-F01Cb

#Pas de sexage
GRE1-2_5b
GRE3-4_1b
GRE3-4_2b
GRE5_2b
GRE6-7_1

3. Sexe list

In [ ]:
INDIV_LIST_FEMALES="Lm-F16b,Lm-,Lm-F13Cbf,Lm-F12Cf,Lm-F09Cf,Lm-F08Cbf,Lm-F07b,Lm-F05Cbf,OR05F,Lm-F02Cb,Lm-F04Cbf,Lm-F06Cbf,OR01F,Lm-F10Cb,Lm-F03Cb,Lm-F14Cbf,Lm-F15Cbf,Lm-F11b,OR03Fb,OR04F,OR06Fb,OR09F,OR08F,OR03M,OR09M,OR02M,OR04Mb,OR05M,Lm-F01Cb,TUNI06-M,TUNI10-Mb"

INDIV_LIST_MALES="Lm-M14Cb,Lm-M11Cb,Lm-M09b,Lm-M15Cb,Lm-M04b,Lm-M05b,Lm-M13Cf,Lm-M12Cb,Lm-M07Cf,Lm-M10b,Lm-M08Cb,Lm-M06Cb,Lm-M01Cf,OR07M,OR10M,OR01M,TUNI13-Fb,TUNI14-Fb,TUNI11-Fb"  

## 2.1 SDPOP WORKFLOW

snakefile

In [ ]:
# L objectif de ce script est prendre en INNPUT un fichier VCF brut (freebayes), de le formater pour SDpop et run SDpop

#######################################
# 1. CONFIG
#######################################

configfile: "config.yaml"

WORKDIR_SDPOP = config["WORKDIR_SDPOP"]
VCF           = config["VCF"]
ref_genome    = config["ref_genome"]
EXONS_BED     = config["EXONS_BED"]

VCF_NORM       = config["VCF_NORM"]
VCF_NORM_FILT  = config["VCF_NORM_FILT"]
VCF_NORM_FILT_EXONS_RECODE = config["VCF_NORM_FILT_EXONS_RECODE"]
CNT_FILE       = config["CNT_FILE"]

SDPOP_NOSEX_OUT = config["SDPOP_NOSEX_OUT"]
SDPOP_XY_OUT    = config["SDPOP_XY_OUT"]
SDPOP_ZW_OUT    = config["SDPOP_ZW_OUT"]

NOSEX_LOG = config["NOSEX_LOG"]
XY_LOG    = config["XY_LOG"]
ZW_LOG    = config["ZW_LOG"]


###########################################
# CIBLES FINALES 
###########################################

rule all:
    input:
        VCF_NORM,
        VCF_NORM_FILT,
        VCF_NORM_FILT_EXONS_RECODE,
        CNT_FILE,
        SDPOP_NOSEX_OUT,
        SDPOP_XY_OUT,
        SDPOP_ZW_OUT,
        NOSEX_LOG,
        XY_LOG,
        ZW_LOG


###############################################
# 3. RUN BCFTOOLS norm 
###############################################

rule bcftools_norm:
    input:
        vcf = VCF,
        ref = ref_genome
    output:
        temp(VCF_NORM)
    threads: 6
    resources:
        mem_mb=48000,
        time="1-00:00:00"
    shell:
        """
        module load bioinfo/Bcftools/1.9

        bcftools norm --threads {threads} {input.vcf} -f {input.ref} -m +any -O v -o {output}
        """


##################################################
# 4. RUN FILTER VCFTOOLS 
##################################################
#Ici on filtre la profondeur minimal a 7 read par individus et on retire l individu Lm-F01Cb

rule vcftools_filter:
    input:
        VCF_NORM=VCF_NORM,
        remove_individuals=config["remove_individuals"]
    output:
        temp(VCF_NORM_FILT)
    threads: 1
    resources:
        mem_mb=8000,
        time="1-00:00:00"
    shell:
        """
        module load bioinfo/samtools/1.14
        module load bioinfo/VCFtools/0.1.16

        vcftools --vcf {input.VCF_NORM} --minDP 7 --remove {input.remove_individuals} --recode --out {output}
        mv {output}.recode.vcf {output}
        """


#########################################################
# 5. CUT EXONS WINDOWS 
#########################################################

rule exons_intersect:
    input:
        vcf = VCF_NORM_FILT,
        bed = EXONS_BED
    output:
        VCF_NORM_FILT_EXONS_RECODE
    threads: 1
    resources:
        mem_mb=8000,
        time="1-00:00:00"
    shell:
        """

        module load devel/python/Python-3.11.1
        module load bioinfo/bedtools/2.30.0

        # extract the header line (CHROM POS ...) of the vcf
        grep -m 1 "CHROM" "{input.vcf}" > "{output}"

        # create an intersect of the vcf and the bed file
        bedtools intersect -a "{input.vcf}" -b "{input.bed}" -wa -wb | \
        awk '{{
            ## We want to retrieve here the last column which corresponds to the name of the gene
            gene_id = $NF
            sub(/-.*/, "", gene_id)

            # print gene_id as the first column 
            printf "%s\t", gene_id

            # Print all the VCF column except the first (#CHROM) and the last 4 (BED column)
            for(i=2;i<=NF-4;i++) printf "%s\t", $i

            print ""
        }}' | sort -k1,1V -k2,2n | uniq >> "{output}"
        # The sort is in order to guarantees one continuous block per gene.
        """


#############################################
# 6. RUN POPSUM 
#############################################

rule popsum_counts:
    input:
        VCF_NORM_FILT_EXONS_RECODE
    output:
        CNT_FILE
    threads: 1
    resources:
        mem_mb=8000,
        time="1-00:00:00"
    shell:
        """
        module load bioinfo/SDpop/f112148

	INDIV_LIST_FEMALES="Lm-F16b,Lm-,Lm-F13Cbf,Lm-F12Cf,Lm-F09Cf,Lm-F08Cbf,Lm-F07b,Lm-F05Cbf,OR05F,Lm-F02Cb,Lm-F04Cbf,Lm-F06Cbf,OR01F,Lm-F10Cb,Lm-F15Cbf,Lm-F03Cb,OR03Fb,OR04F,Lm-F11b,OR06Fb,OR09F,OR08F,OR03M,OR09M,OR02M,OR04Mb,OR05M,Lm-F14Cbf,Lm-F01Cb,TUNI06-M,TUNI10-Mb"
	INDIV_LIST_MALES="Lm-M14Cb,Lm-M11Cb,Lm-M09b,Lm-M15Cb,Lm-M04b,Lm-M05b,Lm-M13Cf,Lm-M12Cb,Lm-M07Cf,Lm-M10b,Lm-M08Cb,Lm-M06Cb,Lm-M01Cf,OR07M,OR10M,OR01M,TUNI13-Fb,TUNI14-Fb,TUNI11-Fb"
        popsum "{input}" "{output}" v "$INDIV_LIST_FEMALES" "$INDIV_LIST_MALES"
        """


#############################################
# 7. RUN SDPOP 
#############################################

rule sdpop_nosex:
    input:
        CNT_FILE
    output:
        SDPOP_NOSEX_OUT
    log:
        NOSEX_LOG
    threads: 1
    resources:
        mem_mb=16000,
        time="1-00:00:00"
    shell:
        """
        module load bioinfo/SDpop/f112148
        sdpop "{input}" "{output}" e n 1 1 s > "{log}"
        """

rule sdpop_xy:
    input:
        CNT_FILE
    output:
        SDPOP_XY_OUT
    log:
        XY_LOG
    threads: 1
    resources:
        mem_mb=16000,
        time="1-00:00:00"
    shell:
        """
        module load bioinfo/SDpop/f112148
        sdpop "{input}" "{output}" e x 1 1 s > "{log}"
        """

rule sdpop_zw:
    input:
        CNT_FILE
    output:
        SDPOP_ZW_OUT
    log:
        ZW_LOG
    threads: 1
    resources:
        mem_mb=16000,
        time="1-00:00:00"
    shell:
        """
        module load bioinfo/SDpop/f112148
        sdpop "{input}" "{output}" e z 1 1 s > "{log}"
        """

run_snakfile.sh

In [ ]:
#!/bin/bash
#SBATCH --job-name=Snakemake_SDPOP__Lnobilis_L_azorica_12_CHR
#SBATCH -p workq
#SBATCH --time=2-00:00:00
#SBATCH --mail-user=tristan.bertrand@umontpellier.fr
#SBATCH --mail-type=ALL
#SBATCH --cpus-per-task=1
#SBATCH --mem=2G
#SBATCH --output=snakemake_%j.log

module load bioinfo/Snakemake/8.20.3

snakemake \
  --snakefile snakefile_SDPOP \
  --cores 6 \
  --rerun-incomplete \
  --keep-going \
  --latency-wait 60 \
  --executor slurm \
  --jobs 4

config.yaml

In [ ]:
WORKDIR_SDPOP: "/home/tbertrand/work/L_nobilis_all_12_CHR/WORKFLOW_SD_POP"
VCF: "/home/tbertrand/work/L_nobilis_all_12_CHR/GENOTYPING/L_nobilis_all_Lazorica_VCFALLSITES_C2_REPMONO_NBESTALLELES2.vcf"
ref_genome: "/home/tbertrand/work/L_nobilis_all_12_CHR/GENOME/Index/GCA_977007225.1_dmLauAzor1_12_CHR.pri_genomic.fna"
EXONS_BED: "/home/tbertrand/work/L_nobilis_all_12_CHR/braker.fixed_ids_exons_sorted.bed"
remove_individuals: "/home/tbertrand/work/L_nobilis_all_12_CHR/WORKFLOW_SD_POP/remove_individuals.txt"

VCF_NORM: "/home/tbertrand/work/L_nobilis_all_12_CHR/SD_POP/L_nobilis_all_Lazorica_norm.vcf"
VCF_NORM_FILT: "/home/tbertrand/work/L_nobilis_all_12_CHR/SD_POP/L_nobilis_all_Lazorica_norm_filtered.vcf"
VCF_NORM_FILT_EXONS_RECODE: "/home/tbertrand/work/L_nobilis_all_12_CHR/SD_POP/L_nobilis_all_Lazorica_norm_filtered_EXONS.recode.vcf"
CNT_FILE: "/home/tbertrand/work/L_nobilis_all_12_CHR/SD_POP/geno_count_L_nobilis_all_Lazorica_norm_filtered_EXONS.recode.out"

SDPOP_NOSEX_OUT: "/home/tbertrand/work/L_nobilis_all_12_CHR/SD_POP/SDPOP_nosexchr_L_nobilis_all_Lazorica_norm_filtered_EXONS.recode.out"
SDPOP_XY_OUT: "/home/tbertrand/work/L_nobilis_all_12_CHR/SD_POP/SDPOP_XY_L_nobilis_all_Lazorica_norm_filtered_EXONS.recode.out"
SDPOP_ZW_OUT: "/home/tbertrand/work/L_nobilis_all_12_CHR/SD_POP/SDPOP_ZW_L_nobilis_all_Lazorica_norm_filtered_EXONS.recode.out"

NOSEX_LOG: "iteration_no_sex_chr_Lnob_POR_Lazo.txt"
XY_LOG: "iteration_XY_sex_chr_Lnob_POR_Lazo.txt"
ZW_LOG: "iteration_ZW_sex_chr_Lnob_POR_Lazo.txt"

## RESULTS

In [ ]:
# Import results Nobilis
scp  tbertrand@genobioinfo2.toulouse.inrae.fr:/home/tbertrand/work/L_nobilis_all_12_CHR/SD_POP/SDPOP_XY_L_nobilis_all_Lazorica_norm_filtered_EXONS.recode.out /Users/cibio/Desktop/Laurus/SDPOP/L_nobilis_all_L_azorica_12CHR/SDPOP
scp  tbertrand@genobioinfo2.toulouse.inrae.fr:/home/tbertrand/work/L_nobilis_all_12_CHR/SD_POP/SDPOP_ZW_L_nobilis_all_Lazorica_norm_filtered_EXONS.recode.out /Users/cibio/Desktop/Laurus/SDPOP/L_nobilis_all_L_azorica_12CHR/SDPOP
scp  tbertrand@genobioinfo2.toulouse.inrae.fr:/home/tbertrand/work/L_nobilis_all_12_CHR/SD_POP/SDPOP_nosexchr_L_nobilis_all_Lazorica_norm_filtered_EXONS.recode.out /Users/cibio/Desktop/Laurus/SDPOP/L_nobilis_all_L_azorica_12CHR/SDPOP


In [ ]:
#L_nobilis_all_12_CHR
grep '>' SDPOP_XY_L_nobilis_all_Lazorica_norm_filtered_EXONS.recode.out > SDPOP_XY_L_nobilis_all_Lazorica_norm_filtered_EXONS.recode.out.contig
sed -i -e "s/ /\t/g" SDPOP_XY_L_nobilis_all_Lazorica_norm_filtered_EXONS.recode.out.contig

grep '>' SDPOP_ZW_L_nobilis_all_Lazorica_norm_filtered_EXONS.recode.out > SDPOP_ZW_L_nobilis_all_Lazorica_norm_filtered_EXONS.recode.out.contig
sed -i -e "s/ /\t/g" SDPOP_ZW_L_nobilis_all_Lazorica_norm_filtered_EXONS.recode.out.contig

grep '>' SDPOP_nosexchr_L_nobilis_all_Lazorica_norm_filtered_EXONS.recode.out > SDPOP_nosexchr_L_nobilis_all_Lazorica_norm_filtered_EXONS.recode.out.contig
sed -i -e "s/ /\t/g" SDPOP_nosexchr_L_nobilis_all_Lazorica_norm_filtered_EXONS.recode.out.contig

In [ ]:
scp -r tbertrand@genobioinfo2.toulouse.inrae.fr:/home/tbertrand/save/annotation .

scp -r tbertrand@genobioinfo2.toulouse.inrae.fr:/home/tbertrand/work/L_novocanariensis .